In [284]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

#for comparisons
from sklearn.linear_model import LinearRegression 
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

### Formula for linear regression:

### $\hat{y}_i = {\beta}_0 + {\beta}_1 X_i$

#### Where,
#### ${\beta}_1 = \sum_{i=1}^{N}\frac{(x_i - \bar{x})(y_i - \bar{y})}{(x_i - \bar{x})^2}$

#### ${\beta}_0 = \bar{y} - {\beta}_1 \bar{x}$

${\beta}_0$ = Intercept of a line <br>
${\beta}_1$ = Slope of the line also called Co-efficent of feature/predictor/independent variable X

In [285]:
class SimpleLinearRegression:
    def __init__(self):
        self.coef_: float = 0
        self.intercept_: float = 0

    def fit(self, X, y):
        # to ensure numpy arrays and there's no type error
        X = np.asarray(X)
        y = np.asarray(y)

        x_mean = np.mean(X)
        y_mean = np.mean(y)

        # B1 = Summation of (x - x_mean) * (y - y_mean) divided by Summation of (square of x - x_mean)
        self.coef_ = np.sum((X - x_mean) * (y - y_mean)) / np.sum((X - x_mean) ** 2)

        #B0 = y_mean - B1 * x_mean
        self.intercept_ = np.mean(y) - self.coef_ * np.mean(X)

    def predict(self, X):
        #because, y_pred = coef_ * X + intercept
        return self.intercept_ + self.coef_ * X
    

def my_mean_absolute_error(y_true, y_pred):
    absolute_error = np.abs(y_true - y_pred)
    return np.sum(absolute_error)/absolute_error.shape[0]

def my_mean_squared_error(y_true, y_pred):
    squared_error = np.square(y_true - y_pred)
    return np.sum(squared_error)/squared_error.shape[0]

def my_root_mean_squared_error(y_true, y_pred):
    return np.sqrt(my_mean_absolute_error(y_true, y_pred))

def my_r2_score(y_true, y_pred):
    y_pred_mean = np.mean(y_true)
    SSM = np.sum((y_true - y_pred_mean) ** 2)
    SSR = np.sum(y_true - y_pred) ** 2
    R2 = 1 - SSR/SSM
    return R2

def adjusted_R2( X, y):
    #implementing the formula 
    # 1- ((1 - R2) * (n - 1) / (n - 1 -k))
    R2 = my_r2_score(X, y)
    n = y.lenght
    k = 1
    ratio_numerator = (1 - R2) * (n-1)
    ratio_denominator = n - 1 - k
    adj_R2 = 1 - ratio_numerator/ratio_denominator
    return adj_R2



In [286]:
df: pd.DataFrame = pd.read_csv('Salary_dataset.csv')

In [287]:
df.sample(5)

,Unnamed: 0,YearsExperience,Salary
26,26,9.6,116970.0
24,24,8.8,109432.0
25,25,9.1,105583.0
8,8,3.3,64446.0
5,5,3.0,56643.0


In [288]:
#removing the ID feature as it's Tote Useless
df = df.drop(columns = 'Unnamed: 0')

In [289]:
df.shape

(30, 2)

In [290]:
#splitting the data into training(25 rows) and testing data(5 rows)
X = df['YearsExperience']
y = df['Salary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 2)

In [291]:
my_model = SimpleLinearRegression()
my_model.fit(X = X_train, y = y_train)
print('My intercept_: ', my_model.intercept_)
print('My coef_: ', my_model.coef_)


My intercept_:  23437.21046340505
My coef_:  9569.586885432866


In [292]:
sklearn_model = LinearRegression()
sklearn_model.fit(X_train.to_numpy().reshape(-1, 1), y_train)
print('Scikit-learn intercept: ', sklearn_model.intercept_)
print('Scikit-learn coef: ', sklearn_model.coef_)

Scikit-learn intercept:  23437.21046340505
Scikit-learn coef:  [9569.58688543]


In [293]:
#Prediction Calculation: Skklearn vs Mine
sklearn_model.predict(X_test.to_numpy().reshape(-1, 1)) - my_model.predict(X_test)

1     0.0
0     0.0
14    0.0
9     0.0
21    0.0
19    0.0
Name: YearsExperience, dtype: float64

In [294]:
#1 MAE:
print("My Mean Absolute Error: ", my_mean_absolute_error(y_true = y_test, y_pred = my_model.predict(X_test.to_numpy())))

print("Mean Absolute Error: ", mean_absolute_error(y_true = y_test, y_pred = my_model.predict(X_test.to_numpy())))

print("Mean Absolute Error of sk:", mean_absolute_error(y_true = y_test, y_pred = sklearn_model.predict(X_test.to_numpy().reshape(-1, 1))))

My Mean Absolute Error:  6802.779572073905
Mean Absolute Error:  6802.779572073905
Mean Absolute Error of sk: 6802.779572073905


In [295]:
#2 MSE 
print("Mean Squared of my class: ", mean_squared_error(y_test, my_model.predict(X_test.to_numpy())))
print("Mean Squared of sk class: ", mean_squared_error(y_test, sklearn_model.predict(X_test.to_numpy().reshape(-1, 1))))

Mean Squared of my class:  56137509.99782566
Mean Squared of sk class:  56137509.99782566


In [296]:
print(f"Root Mean Squared Error of my class: {root_mean_squared_error(y_test, my_model.predict(X_test.to_numpy()))}")
print(f"Root Mean Squared Error of scikit learn: {root_mean_squared_error(y_test, sklearn_model.predict(X_test.to_numpy().reshape(-1, 1)))}")

Root Mean Squared Error of my class: 7492.49691343451
Root Mean Squared Error of scikit learn: 7492.49691343451


In [297]:
#4. R2 Score
print("My R2 Score: ", r2_score(y_true = y_test.to_numpy(), y_pred = my_model.predict(X_test.to_numpy())))

print("Sk R2 Score: ", r2_score(y_true = y_test.to_numpy(), y_pred = sklearn_model.predict(X_test.to_numpy().reshape(-1, 1))))

My R2 Score:  0.8886956733784561
Sk R2 Score:  0.8886956733784561


In [298]:
print("Scikit-Learn r2 score: ", my_r2_score(y_true = y_test.to_numpy(), y_pred = sklearn_model.predict(X_test.to_numpy().reshape(-1, 1))))


print("Scikit-Learn r2 score: ", r2_score(y_true = y_test.to_numpy(), y_pred = sklearn_model.predict(X_test.to_numpy().
reshape(-1, 1))))

Scikit-Learn r2 score:  0.8266659445915482
Scikit-Learn r2 score:  0.8886956733784561
